# 015 - QAOA via QPU (V 1.1)  

Drafted 3/11/26 - Not Run Yet

### Admin

In [ ]:
# Data Utilites 
import numpy as np

# Evironmental Libraries 
from tabulate import tabulate
import environment as envir

# Qiskit model libraries 
from qiskit_algorithms.utils import algorithm_globals
from qiskit_algorithms import SamplingVQE
from qiskit_algorithms.optimizers import COBYLA

from qiskit.circuit.library import n_local
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_finance.applications.optimization import PortfolioOptimization

# IBM Quantum runtime & sampler 
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler


In [ ]:
data, headers = envir.environment_state()
print(tabulate(data, headers=headers))

### Notebook Parameters 

In [ ]:
# Set Seed 
algorithm_globals.random_seed = 1234

# Print (debug) 
print_flag = True 

### The Data 

In [ ]:
num_assets = 4

expected_returns = np.array([0.10, 0.20, 0.15, 0.18])

covariances = np.array([
    [0.005, -0.010, 0.004, -0.002],
    [-0.010, 0.040, -0.002, 0.004],
    [0.004, -0.002, 0.023, 0.002],
    [-0.002, 0.004, 0.002, 0.018],
])

risk_factor = 0.5
budget = 2

### The Model

In [ ]:
portfolio = PortfolioOptimization(
    expected_returns,
    covariances,
    risk_factor,
    budget
)

qp = portfolio.to_quadratic_program()

if print_flag: 
    print(qp.prettyprint())

In [ ]:
ansatz = n_local(
    num_qubits=num_assets,
    rotation_blocks=["ry"],
    entanglement_blocks="cz",
    entanglement="full",
    reps=2
)

In [ ]:
# Classical Optimizer 
optimizer = COBYLA(maxiter=250)

In [ ]:
# Make sure you have your IBM Quantum API token saved in qiskit-ibm-runtime
service = QiskitRuntimeService(channel="ibm_quantum")  

# Select a small device; change as needed
backend = service.backend("ibm_perth")  

In [ ]:
# Create the Sampler 
sampler = Sampler(options={"backend": backend, "shots": 8192})

In [ ]:
vqe = SamplingVQE(
    sampler=sampler,
    ansatz=ansatz,
    optimizer=optimizer
)

### Solve 

In [ ]:
result = solver.solve(qp)

### Post Processing 

In [ ]:
print("Optimal portfolio:", result.x)
print("Optimal value:", result.fval)

# Optional: human-readable
for i, val in enumerate(result.x):
    if val == 1:
        print(f"Invest in asset {i}")